## Question 2

In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

In [4]:
## Import data
df = pd.read_csv('spambase/spambase.data', header=None)

columns = [
    'word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d',
    'word_freq_our', 'word_freq_over', 'word_freq_remove', 'word_freq_internet',
    'word_freq_order', 'word_freq_mail', 'word_freq_receive', 'word_freq_will',
    'word_freq_people', 'word_freq_report', 'word_freq_addresses', 'word_freq_free',
    'word_freq_business', 'word_freq_email', 'word_freq_you', 'word_freq_credit',
    'word_freq_your', 'word_freq_font', 'word_freq_000', 'word_freq_money',
    'word_freq_hp', 'word_freq_hpl', 'word_freq_george', 'word_freq_650',
    'word_freq_lab', 'word_freq_labs', 'word_freq_telnet', 'word_freq_857',
    'word_freq_data', 'word_freq_415', 'word_freq_85', 'word_freq_technology',
    'word_freq_1999', 'word_freq_parts', 'word_freq_pm', 'word_freq_direct',
    'word_freq_cs', 'word_freq_meeting', 'word_freq_original', 'word_freq_project',
    'word_freq_re', 'word_freq_edu', 'word_freq_table', 'word_freq_conference',
    'char_freq_;', 'char_freq_(', 'char_freq_[', 'char_freq_!',
    'char_freq_$', 'char_freq_#',
    'capital_run_length_average', 'capital_run_length_longest',
    'capital_run_length_total', 'spam'
]

df.columns = columns

In [5]:
## Split data for training

X = df.drop(columns="spam")
y = df['spam']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1
)

In [6]:
## Scale data

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

In [7]:
## Define GD for logistic regression

class LogisticRegressionGD:
    
    def __init__(self, alpha=0.01, iterations=100):
        self.alpha = alpha
        self.iterations = iterations
        self.theta = None
        self.losses = {}  
        
    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    
    def _cross_entropy_loss(self, y, y_hat):
        eps = 1e-15
        y_hat = np.clip(y_hat, eps, 1 - eps)
        return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
        
    def fit(self, X, y, report_at={10, 50, 100}):
        n = X.shape[0]
        
        ones = np.ones((n, 1))
        X_b = np.hstack((ones, X))
        self.theta = np.zeros(X_b.shape[1])
        
        for i in range(1, self.iterations + 1):
            y_hat = self._sigmoid(X_b @ self.theta)
            errors = y_hat - y
            
            # Gradient
            gradient = (1/n) * (errors.T @ X_b)
            self.theta -= self.alpha * gradient
            
            if i in report_at:
                self.losses[i] = self._cross_entropy_loss(y, self._sigmoid(X_b @ self.theta))
            
    def predict_proba(self, X):
        n = X.shape[0]
        ones = np.ones((n, 1))
        X_b = np.hstack((ones, X))
        return self._sigmoid(X_b @ self.theta)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


In [11]:
## Define learning rates and train models

learning_rates = [0.01, 0.1, 0.5]
report_iters   = {10, 50, 100}
models_gd      = {}

for lr in learning_rates:
    m = LogisticRegressionGD(alpha=lr, iterations=100)
    m.fit(X_train_s, y_train, report_at=report_iters)
    models_gd[lr] = m

In [12]:
## Print Cross-Entropy loss for each value of iteration and learning rate
print("Cross-Entropy Loss")
for lr in learning_rates:
    print(f"\n  lr={lr}")
    for it in [10, 50, 100]:
        print(f"    iter {it}: {models_gd[lr].losses[it]:.6f}")


Cross-Entropy Loss

  lr=0.01
    iter 10: 0.651430
    iter 50: 0.542565
    iter 100: 0.470134

  lr=0.1
    iter 10: 0.466798
    iter 50: 0.325520
    iter 100: 0.289150

  lr=0.5
    iter 10: 0.321613
    iter 50: 0.257671
    iter 100: 0.242496


In [13]:
## Metrics after 100 iterations

print("\nMetrics at 100 Iterations")
for split_name, X_s, y_arr in [('Train', X_train_s, y_train),
                                 ('Test',  X_test_s,  y_test)]:
    print(f"\n  {split_name} Set")
    for lr in learning_rates:
        y_pred = models_gd[lr].predict(X_s)
        print(f"    lr={lr}  Acc={accuracy_score(y_arr, y_pred):.4f}  "
              f"Prec={precision_score(y_arr, y_pred):.4f}  "
              f"Rec={recall_score(y_arr, y_pred):.4f}  "
              f"F1={f1_score(y_arr, y_pred):.4f}")


Metrics at 100 Iterations

  Train Set
    lr=0.01  Acc=0.8939  Prec=0.8728  Rec=0.8562  F1=0.8644
    lr=0.1  Acc=0.9055  Prec=0.9092  Rec=0.8452  F1=0.8760
    lr=0.5  Acc=0.9194  Prec=0.9248  Rec=0.8665  F1=0.8947

  Test Set
    lr=0.01  Acc=0.9192  Prec=0.9260  Rec=0.8622  F1=0.8930
    lr=0.1  Acc=0.9114  Prec=0.9416  Rec=0.8244  F1=0.8791
    lr=0.5  Acc=0.9235  Prec=0.9480  Rec=0.8511  F1=0.8970


In [19]:
## Train and test sklearn Logistic Regression to compare:

lr = LogisticRegression(max_iter=5000)
lr.fit(X_train_s, y_train)

for split_name, X_s, y_s in [('Train', X_train_s, y_train), ('Test', X_test_s, y_test)]:
    y_pred = lr.predict(X_s)
    print(f"{split_name} Set")
    print(f"  Acc={accuracy_score(y_s, y_pred):.4f}  Prec={precision_score(y_s, y_pred):.4f}  Rec={recall_score(y_s, y_pred):.4f}  F1={f1_score(y_s, y_pred):.4f}")

Train Set
  Acc=0.9296  Prec=0.9262  Rec=0.8929  F1=0.9092
Test Set
  Acc=0.9279  Prec=0.9400  Rec=0.8711  F1=0.9043


## Comparison

Overall, the sklearn model performs better than our manual model. It produces a higher accuracy on the train and test set, a higher precision on train and only a slightly lower (<0.01) on test, and higher recall and F1 on both. Looking at just the different learning rates, 0.1 performs the best on average and with the closest values between training and testing, but the package model is still just better. Maybe with more iterations or different learning values the manual model could out perform but that would require more testing. 